In [44]:
%%configure -f
{
    "jars": ["https://repo1.maven.org/maven2/org/datasyslab/geotools-wrapper/1.9.1-33.5/geotools-wrapper-1.9.1-33.5.jar", "https://repo1.maven.org/maven2/org/apache/sedona/sedona-spark-shaded-4.1_2.13/1.9.1/sedona-spark-shaded-4.1_2.13-1.9.1.jar"]
}

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, -1, Finished, Available, Finished, True)

In [45]:
from pyspark.sql.functions import *
from sedona.spark import SedonaContext

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 5, Finished, Available, Finished, False)

In [46]:
def read_table(table_name):
    return spark.read.table(f"Bronze_lakehouse.dbo.{table_name}")

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 6, Finished, Available, Finished, False)

In [47]:
def save_table(df, table_name):
    df.write.format("delta").mode("overwrite").saveAsTable(f"Silver_lakehouse.dbo.{table_name}")

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 7, Finished, Available, Finished, False)

# Process yellow taxi data

In [52]:
YELLOW_TAXI_FOLDER_PATH = "Files/NYC_Taxi/yellow_tripdata"

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 12, Finished, Available, Finished, False)

In [53]:
YELLOW_TAXI_SCHEMA = {
    "pickup_datetime": "timestamp",
    "dropoff_datetime": "timestamp",
    "passenger_count": "integer",
    "trip_distance": "double",
    "pickup_location_id": "integer",
    "dropoff_location_id": "integer",
    "pickup_longitude": "double",
    "pickup_latitude": "double",
    "dropoff_longitude": "double",
    "dropoff_latitude": "double",
    "fare_amount": "double",
    "total_amount": "double",
    "tip_amount": "double"
}

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 13, Finished, Available, Finished, False)

In [54]:
YELLOW_TAXI_COLUMN_MAPPING = {
    "Trip_Pickup_DateTime": "pickup_datetime",
    "tpep_pickup_datetime": "pickup_datetime",
    "Trip_Dropoff_DateTime": "dropoff_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime",
    "Passenger_Count": "passenger_count",
    "Trip_Distance": "trip_distance",
    "PULocationID": "pickup_location_id",
    "DOLocationID": "dropoff_location_id",
    "Start_Lon": "pickup_longitude",
    "Start_Lat": "pickup_latitude",
    "End_Lon": "dropoff_longitude",
    "End_Lat": "dropoff_latitude",
    "Fare_Amt": "fare_amount",
    "Total_Amt": "total_amount",
    "Tip_Amt": "tip_amount"
}

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 14, Finished, Available, Finished, False)

In [55]:
TAXI_ZONES_SHAPEFILE_FOLDER_PATH = "Files/NYC_Taxi/taxi_zones/"
TAXI_ZONES_SHAPEFILE_PATH = "Files/NYC_Taxi/taxi_zones/taxi_zones.shp"

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 15, Finished, Available, Finished, False)

In [56]:
def get_parquet_files(folder_path):
    files = notebookutils.fs.ls(folder_path)
    return [file.path for file in files if file.name.endswith(".parquet")]

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 16, Finished, Available, Finished, False)

In [57]:
def read_parquet_file(file_path):
    return spark.read.parquet(file_path)

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 17, Finished, Available, Finished, False)

In [58]:
def read_parquet_files(folder_path):
    file_paths = get_parquet_files(folder_path)
    return [read_parquet_file(file_path) for file_path in file_paths]

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 18, Finished, Available, Finished, False)

In [59]:
def rename_columns(df, column_mapping):
    for old_name, new_name in column_mapping.items():
        if old_name in df.columns:
            df = df.withColumnRenamed(old_name, new_name)
    return df

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 19, Finished, Available, Finished, False)

In [60]:
def add_missing_columns(df, required_columns):
    for column, data_type in required_columns.items():
        if column not in df.columns:
            df = df.withColumn(column, lit(None).cast(data_type))
    return df

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 20, Finished, Available, Finished, False)

In [61]:
def cast_columns(df, schema):
    for column, data_type in schema.items():
        df = df.withColumn(column, col(column).cast(data_type))
    return df

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 21, Finished, Available, Finished, False)

In [62]:
def select_columns(df, schema):
    return df.select(*schema.keys())

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 22, Finished, Available, Finished, False)

In [63]:
def normalize_yellow_taxi(df):
    df = rename_columns(df, YELLOW_TAXI_COLUMN_MAPPING)
    df = add_missing_columns(df, YELLOW_TAXI_SCHEMA)
    df = cast_columns(df, YELLOW_TAXI_SCHEMA)
    df = select_columns(df, YELLOW_TAXI_SCHEMA)
    return df

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 23, Finished, Available, Finished, False)

In [64]:
def combine_dataframes(dataframes):
    result = dataframes[0]
    for df in dataframes[1:]:
        result = result.unionByName(df)
    return result

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 24, Finished, Available, Finished, False)

In [65]:
def transform_yellow_taxi(folder_path):
    dfs = read_parquet_files(folder_path)
    normalized_dfs = [normalize_yellow_taxi(df) for df in dfs]
    return combine_dataframes(normalized_dfs)

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 25, Finished, Available, Finished, False)

In [66]:
def validate_yellow_taxi(df):
    assert not df.isEmpty(), "Table is empty"
    assert df.columns == list(YELLOW_TAXI_SCHEMA.keys()), "Table has an unexpected schema"

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 26, Finished, Available, Finished, False)

Convert coordinates to zones 

In [67]:
def create_sedona_context(spark):
    return SedonaContext.create(spark)

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 27, Finished, Available, Finished, False)

In [74]:
def read_taxi_zones_shapefile(sedona, file_path):
    return sedona.read.format("org.apache.sedona.sql.datasources.shapefile.ShapefileDataSource").load(file_path)

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 34, Finished, Available, Finished, False)

In [69]:
def transform_taxi_zones(zones_df):
    return zones_df.selectExpr("*", "ST_Transform(geometry, 'EPSG:2263', 'EPSG:4326') AS new_geometry")

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 29, Finished, Available, Finished, False)

In [70]:
def create_temp_view(df, view_name):
    df.createOrReplaceTempView(view_name)

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 30, Finished, Available, Finished, False)

In [71]:
def enrich_yellow_taxi_locations(df, sedona):
    create_temp_view(df, "trips")
    return sedona.sql("""
        SELECT
            trips.pickup_datetime,
            trips.dropoff_datetime,
            trips.passenger_count,
            trips.trip_distance,
            COALESCE(trips.pickup_location_id, pickup_zone.LocationID) AS pickup_location_id,
            COALESCE(trips.dropoff_location_id, dropoff_zone.LocationID) AS dropoff_location_id,
            trips.fare_amount,
            trips.total_amount,
            trips.tip_amount
        FROM trips
        LEFT JOIN zones AS pickup_zone 
        ON ST_Contains(pickup_zone.new_geometry, ST_Point(trips.pickup_longitude, trips.pickup_latitude))
        LEFT JOIN zones AS dropoff_zone
        ON ST_Contains(dropoff_zone.new_geometry, ST_Point(trips.dropoff_longitude, trips.dropoff_latitude))
    """)

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 31, Finished, Available, Finished, False)

In [72]:
def process_yellow_taxi(folder_path, shapefile_path):
    sedona = create_sedona_context(spark)
    zones_df = read_taxi_zones_shapefile(sedona, shapefile_path)
    zones_df = transform_taxi_zones(zones_df)
    create_temp_view(zones_df, "zones")
    df = transform_yellow_taxi(folder_path)
    validate_yellow_taxi(df)
    result_df = enrich_yellow_taxi_locations(df, sedona)
    save_table(result_df, "yellow_taxi")

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 32, Finished, Available, Finished, False)

In [75]:
process_yellow_taxi(YELLOW_TAXI_FOLDER_PATH, TAXI_ZONES_SHAPEFILE_FOLDER_PATH)

StatementMeta(, 9dc0a1c3-ce85-4cb1-94d3-d498f3c2959e, 35, Submitted, Running, Running, True)